<a href="https://colab.research.google.com/github/martine-augustin/Marionnaud/blob/main/Copie_de_marionnaud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Marionnaud projet


## Variable et installation

In [4]:
# Install
!pip install beautifulsoup4
!pip install lxml




In [5]:
# Import
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import time
import random
import csv
import pandas as pd

In [6]:
#Constante
id_avis = 0
page_avis = 0
page_avis_max = 365
page_avis_end = False
liste_des_avis = list()

## Fonction

In [7]:
def unAvis(section):
    global id_avis

    img_star = section.find('img', class_='CDS_StarRating_starRating__614d2e')
    if not img_star:
        return None

    nb_etoile_text = img_star.get('alt')
    nb_etoile = nb_etoile_text[5]

    date_commentaire_tag = section.find('span')
    if not date_commentaire_tag or not nb_etoile.isdigit():
        return None

    date_commentaire = date_commentaire_tag.get_text(strip=True)
    if not date_commentaire[0].isdigit():
        return None

    id_avis += 1
    titre_tag = section.find('h2')
    titre = titre_tag.get_text(strip=True) if titre_tag else None

    commentaire_tag = section.find('p')
    commentaire_text = commentaire_tag.get_text(strip=True) if commentaire_tag else None

    return [id_avis, page_avis, nb_etoile, date_commentaire, titre, commentaire_text]

In [8]:
def PageAvis(url):
    liste_des_avis_page = []

    # Parser le HTML
    soup = BeautifulSoup(response.text, 'html.parser')
    sections = soup.find_all('section')  # toutes les sections

    for section in sections:
        avis = unAvis(section)
        if avis:  # si unAvis renvoie bien une liste
            liste_des_avis_page.append(avis)

    return liste_des_avis_page

In [9]:
def CreateCSV(tab_avis):
  with open("avis.csv", mode="w", newline="", encoding="utf-8") as fichier_csv:
    writer = csv.writer(fichier_csv)

    # Écrire une ligne d'en-tête
    writer.writerow(["Avis ID", "Page ID", "Nombre etoile", "Date", "Titre", "Commentaire"])

    # Écrire les données
    for ligne in tab_avis:
        writer.writerow(ligne)

In [10]:
def OpenCSV(filename):
    """
    Ouvre un fichier CSV et retourne un DataFrame pandas
    """
    try:
        df = pd.read_csv(filename, encoding="utf-8")
        return df
    except FileNotFoundError:
        print(f"Erreur : le fichier {filename} n'existe pas.")
        return None
    except pd.errors.EmptyDataError:
        print(f"Erreur : le fichier {filename} est vide.")
        return None

## Scrapping

In [ ]:
!pip

In [3]:
import time
import pandas as pd
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

# Installation auto du driver
chromedriver_autoinstaller.install()

options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')

driver = webdriver.Chrome(options=options)

url = "https://fr.trustpilot.com/review/www.marionnaud.com"
driver.get(url)
time.sleep(5)

all_data = []
page_count = 0
MAX_PAGES = 5 # Ajustez selon vos besoins (366 pour la totalité)

try:
    while page_count < MAX_PAGES:
        page_count += 1
        print(f"📄 Scraping page {page_count}...", end="\r")

        # Parser la page actuelle
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Trouver tous les containers d'avis
        reviews = soup.find_all('section', class_='styles_reviewContentwrapper__K2aRu')

        for review in reviews:
            # 1. Extraction du texte de l'opinion
            content = review.find('p', class_='typography_body-l__KUYFJ')
            text = content.text.strip() if content else "Pas de texte"

            # 2. Extraction de la note (étoiles)
            # Trustpilot stocke la note dans l'attribut data-rating de l'image de l'étoile
            rating_div = review.find('div', class_='styles_reviewHeader__iU9Px')
            rating = rating_div.find('img')['alt'] if rating_div and rating_div.find('img') else "N/A"

            # 3. Extraction de la date
            date_element = review.find('time')
            date_val = date_element['datetime'] if date_element else "N/A"

            all_data.append({
                "Date": date_val,
                "Note": rating,
                "Avis": text
            })

        # --- Gestion de la pagination ---
        try:
            # On cherche le lien "Page suivante" (il contient souvent l'attribut name="pagination-button-next")
            next_button = driver.find_element(By.CSS_SELECTOR, "a[name='pagination-button-next']")

            # Vérifier si le bouton est désactivé (fin des pages)
            if not next_button.is_enabled() or next_button.get_attribute("aria-disabled") == "true":
                break

            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(3) # Pause cruciale pour laisser le DOM se mettre à jour
        except:
            print("\nFin des pages ou bouton suivant introuvable.")
            break

finally:
    driver.quit()

# Sauvegarde
df = pd.DataFrame(all_data)
df.to_csv("avis_marionnaud.csv", index=False, encoding='utf-8-sig')
print(f"\n🎉 Terminé ! {len(all_data)} avis récupérés et sauvegardés dans 'avis_marionnaud.csv'.")

ModuleNotFoundError: No module named 'chromedriver_autoinstaller'

In [11]:
#On recupere les avis de toutes les pages Marionnaud
page_avis= 0
page_avis_end= False

while page_avis_end == False:
  page_avis += 1
  url = 'https://fr.trustpilot.com/review/www.marionnaud.com?page=' + str(page_avis)

  # Récupérer le contenu de la page avec requests
  response = requests.get(url)

  if response.status_code != 200:
    print(f"Error status code : {response.status_code}")
    page_avis_end = True
  else:
    result = PageAvis(url)
    if len(result)> 0:
      if liste_des_avis:
        liste_des_avis = liste_des_avis + result
      else:
        liste_des_avis = result
    else:
      page_avis_end= True

  # Génère un nombre entier aléatoire entre 2 et 5
  delai = random.randint(2, 5)
  print("Début du timeur de la page : " + str(page_avis) + " en cours ...")
  time.sleep(delai)  # attend 5 secondes

  print("--------------------------")
  print("url :" + url)
  print(result)
  print("nombre élément liste_des_avis : " + str(len(liste_des_avis)))

print(liste_des_avis)
CreateCSV(liste_des_avis)

Début du timeur de la page : 1 en cours ...
--------------------------
url :https://fr.trustpilot.com/review/www.marionnaud.com?page=1
[[1, 1, '5', '27 février 2026', 'BONJOUR,', "BONJOUR,JE SUIS CONTENTE DE MON PASSAGE A LA BOUTIQUE MARIONNAUD RUE ORDENER , PERSONNELS COMPETENTS , SEVIABLES , AIMABLES , ET AUSSI D'UNE GENTILLESSE SANS PAREIL.JE RECOMMANDE CETTE BOUTIQUE.FORNAS MICHELE"], [2, 1, '5', '24 février 2026', 'Un parfum pour mon petit fils.', "Un parfum pour mon petit fils.C'était une première pour lui, j'avais besoin de conseils et j'ai trouvé une équipe formidable au magasin d'Antony. Ce magasin m'a fait changé d'idée sur Marionnaud (en effet, tous les magasins ne sont pas aussi aimables).BRAVO à cette équipe"], [3, 1, '1', '28 février 2026', 'J’ai passé une commande sur leur site…', 'J’ai passé une commande sur leur site 21/02/2026 , expédiée par Marionnaud. Trois dates de livraison m’ont été données via le transporteur Colissimo, mais le colis n’est jamais arrivé. J’ai dû

In [12]:
# Ouvrir le CSV créé précédemment
df_avis = OpenCSV("avis.csv")

# Vérifier que ça a fonctionné
if df_avis is not None:
    print(df_avis.head())  # affiche les 5 premières lignes


   Avis ID  Page ID  Nombre etoile             Date  \
0        1        1              5  27 février 2026   
1        2        1              5  24 février 2026   
2        3        1              1  28 février 2026   
3        4        1              5  23 février 2026   
4        5        1              5  25 février 2026   

                                    Titre  \
0                                BONJOUR,   
1          Un parfum pour mon petit fils.   
2  J’ai passé une commande sur leur site…   
3                                Le choix   
4                       Une équipe au top   

                                         Commentaire  
0  BONJOUR,JE SUIS CONTENTE DE MON PASSAGE A LA B...  
1  Un parfum pour mon petit fils.C'était une prem...  
2  J’ai passé une commande sur leur site 21/02/20...  
3  Le choix, l accueil et les conseils sont les a...  
4  Une équipe au top! De très bons conseils et un...  
